In [63]:
from multimodal_interactor import  MultimodalQAInteractor, MultimodalNote, MultimodalAnnotationInteractor
from demonstrations import Demonstrations
from input_output_utils import load_taxonomy
from vllm.utils import FlexibleArgumentParser
from vllms import VLM
import yaml
import os
# include /Users/knf792/gits/MultimodalTaxonomy im the namepsace
os.sys.path.append('/Users/knf792/gits/MultimodalTaxonomy')
from transformers import AutoTokenizer, AutoProcessor, AutoModelForVision2Seq
from vllm import LLM, EngineArgs, SamplingParams
from vllm.lora.request import LoRARequest
from vllm.multimodal.image import convert_image_mode
from vllm.sampling_params import GuidedDecodingParams
from dataclasses import asdict


In [2]:
def parse_args():
    parser = FlexibleArgumentParser(
        description="Demo on using vLLM for offline inference with "
        "vision language models that support multi-image input for text "
        "generation"
    )
    parser.add_argument(
        "--model-name",
        "-m",
        type=str,
        default="gemma3",
        # choices=VLM.models.keys(),
        help='Huggingface "model_type".',
    )
    parser.add_argument(
        "--notes-path",
        type=str,
        help="path to the directory containing the images",
        default="data/tweets_with_images.csv",
    )
    parser.add_argument(
        "--prompt-path",
        type=str,
        help="path to the directory containing the images",
        default="prompts/flowchart_prompt_annotations.txt",
    )
    parser.add_argument(
        "--debug-mode", action="store_true", help="Whether to load the model or not. "
    )
    parser.add_argument(
        "--taxonomy-path",
        type=str,
        help="path to the taxonomy file",
        default="prompts/full_taxonomy.json",
    )
    parser.add_argument(
        "--taxonomy-level",
        nargs="+",
        type=str,
        default=["multimodal_taxonomy"],
        help="The taxonomy level to apply. Currently only 'type' and 'subtype' are supported.",
    )
    parser.add_argument(
        "--image-path",
        type=str,
        help="path to the directory containing the images",
        default="/home/knf792/gits/MMFC-cnotes/data/tweet_images/",
    )
    parser.add_argument(
        "--method",
        type=str,
        default="generate",
        choices=["generate", "chat"],
        help="The method to run in `vllm.LLM`.",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=None,
        help="Set the seed when initializing `vllm.LLM`.",
    )
    parser.add_argument(
        "--task",
        type=str,
        default="baseline",
        choices=["baseline", "flowchart"],
        help="The task to run. Currently only 'type_analysis' is supported.",
    )
    parser.add_argument(
        "--temperature",
        "-t",
        type=float,
        default=0.0,
        help="The temperature to use for sampling. 0.0 means greedy decoding.",
    )
    parser.add_argument(
        "--zero-shot",
        action="store_true",
        help="Whether to run the model in zero-shot mode. ",
    )
    parser.add_argument(
        "--chat",
        action="store_true",
        help="Whether to include a system prompt in the chat template.",
    )
    parser.add_argument(
        "--save-path",
        type=str,
        help="path to where to save the results",
        default="results/",
    )
    parser.add_argument(
        "--batch-size",
        type=int,
        default=32,
        help="Batch size for processing images. This is useful for large datasets.",
    )
    parser.add_argument(
        "--max-tokens",
        type=int,
        default=1024,
        help="Maximum number of tokens to generate.",
    )
    parser.add_argument(
        "--max-samples",
        type=int,
        default=-1,
        help="Maximum samples to load.",
    )
    parser.add_argument(
        "--num-demos",
        type=int,
        default=4,
        help="Number of demonstrations to include in the prompt.",
    )
    parser.add_argument(
        "--demonstration-type",
        type=str,
        default="same_single",
        choices=["same_single", "same_flow", "all", "random", "random_2", "random_3", "random_4", "random_5"],
        help="Type of demonstrations to include in the prompt.",
    )
    parser.add_argument(
        "--demo-data-path",
        type=str,
        help="Path to the demonstration data CSV file.",
        default="data/qualification_dataset_en.csv",
    )
    parser.add_argument(
        "--demo-images-path",
        type=str,
        help="Path to the demonstration images directory.",
        default="data/qualification_images/",
    )
    return parser.parse_args(args=["--taxonomy_path", "../prompts/taxonomy_annoation_experiment.yaml", 
                                   "--demo-images-path", "../data/qualification_images/",
                                   "--demo-data-path", "../data/qualification_dataset_en.csv",
                                   "--prompt-path", "../prompts/flowchart_prompt_annotations.txt",
                                   "--demonstration-type", "same_flow",
                                   "--model-name", "smolvlm",])

args = parse_args()
args

Namespace(model_name='smolvlm', notes_path='data/tweets_with_images.csv', prompt_path='../prompts/flowchart_prompt_annotations.txt', debug_mode=False, taxonomy_path='../prompts/taxonomy_annoation_experiment.yaml', taxonomy_level=['multimodal_taxonomy'], image_path='/home/knf792/gits/MMFC-cnotes/data/tweet_images/', method='generate', seed=None, task='baseline', temperature=0.0, zero_shot=False, chat=False, save_path='results/', batch_size=32, max_tokens=1024, max_samples=-1, num_demos=4, demonstration_type='same_flow', demo_data_path='../data/qualification_dataset_en.csv', demo_images_path='../data/qualification_images/')

In [3]:
taxonomy = load_taxonomy(args.taxonomy_path)
note = MultimodalNote(
    user='Ordnance Arbiter.',
    note="The Americans joined the war right at the end of 1941",
    post="@mikenelson586 Wondering how many Brits there were",
    image_path="'/Users/knf792/Documents/danish-vocab-extention copy/icon.png'",
)

interactor = MultimodalAnnotationInteractor(note=note, args=args)
vlm = VLM(args=args)


KeyError: 'smolvlm'

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

<|im_start|>System: You are an expert fact-checking system that specialises in annotating multimodal misinformation that is spread in social media. Your goal is to analyse a <user, post, image, fact-check verdict> tuple and answer a set of questions regarding the relationship between the post and the image, and the different ways in which the image is used to promote misinformation. You must answer each question to the best of your ability.<end_of_utterance>
User: Instructions:
You are an expert fact-checking system that specialises in annotating multimodal misinformation from social media. 
You will be given a <user, post, image, fact-check verdict> tuple, where <post, image> are the text and image posted by <user> on a social media platform that contain some claim, and <fact-check verdict> is a fact-checking verdict determining whether the <post, image> spread misinformation, and why. 
Your goal is to analyse the <post, image, fact-check verdict> triplet, and answer a set of question

In [67]:


# Preparation for inference
# model = AutoModelForVision2Seq.from_pretrained(
#     "HuggingFaceTB/SmolVLM-256M-Instruct"
# )
llm = LLM(
    **asdict(EngineArgs(
        model="HuggingFaceTB/SmolVLM-256M-Instruct",
        seed=args.seed,
        max_model_len=1024
    ))
)



WARNING 01-14 17:56:35 [config.py:3392] Your device 'cpu' doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 01-14 17:56:35 [config.py:3443] Casting torch.bfloat16 to torch.float16.
INFO 01-14 17:56:35 [config.py:1604] Using max model len 1024
INFO 01-14 17:56:35 [arg_utils.py:1030] Chunked prefill is not supported for ARM and POWER CPUs; disabling it for V1 backend.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 01-14 17:56:42 [__init__.py:235] Automatically detected platform cpu.
WARNING 01-14 17:56:46 [_custom_ops.py:20] Failed to import from vllm._C with ImportError("dlopen(/Users/knf792/gits/MultimodalTaxonomy/.venv/lib/python3.12/site-packages/vllm/_C.abi3.so, 0x0002): symbol not found in flat namespace '__Z14int8_scaled_mmRN2at6TensorERKS0_S3_S3_S3_RKNSt3__18optionalIS0_EE'")
INFO 01-14 17:56:46 [core.py:572] Waiting for init message from front-end.
INFO 01-14 17:56:46 [core.py:71] Initializing a V1 LLM engine (v0.10.0) with config: model='HuggingFaceTB/SmolVLM-256M-Instruct', speculative_config=None, tokenizer='HuggingFaceTB/SmolVLM-256M-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eag

Process EngineCore_0:
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/miniconda3/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/knf792/gits/MultimodalTaxonomy/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 636, in run_engine_core
    raise e
  File "/Users/knf792/gits/MultimodalTaxonomy/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 623, in run_engine_core
    engine_core = EngineCoreProc(*args, **kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/knf792/gits/MultimodalTaxonomy/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 441, in __init__
    super().__init__(vllm_config, executor_class, log_stats,
  File "/Users/knf792/gits/MultimodalTaxonomy/.venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 77, in __init__
    sel

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {'EngineCore_0': 1}

In [60]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "And here is Image 2:"},
            {
                "type": "image",
                "image": "https://en.wikipedia.org/wiki/Cat#/media/File:Cat_August_2010-4.jpg",
            },
            {"type": "text", "text": "How many images do you see and what are they?"},
        ],
    }
]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=["https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg", "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"], return_tensors="pt")

out = model.generate(**inputs, max_new_tokens=300)
generated_texts = processor.batch_decode(
    out,
    skip_special_tokens=True,
)

print(generated_texts[0])

User:



And here is Image 2:



How many images do you see and what are they?
Assistant: There are two images in the image.


In [46]:
processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

'<|im_start|>User:<image>And here is Image 2:<image>Describe the two images briefly.<end_of_utterance>\nAssistant:'

In [4]:
multimodal_interactor.update_next_turn(["yes"])
multimodal_interactor.update_next_turn(["no"])


In [5]:
multimodal_interactor.update_next_turn(["yes"])
multimodal_interactor.messages


[{'role': 'system',
  'content': [{'type': 'text',
    'text': 'You are an expert fact-checking system that specialises in annotating multimodal misinformation that is spread in social media. Your goal is to analyse a <post, image, fact-check verdict> triplet and answer a set of questions regarding the relationship between the post and the image, and the different ways in which the image is used to promote misinformation. You must answer each question to the best of your ability.'}]},
 {'role': 'user',
  'content': [{'type': 'text',
    'text': '<instructions>\nYou are an expert fact-checking system that specialises in annotating multimodal misinformation from social media. \nYou will be given a <post, image, fact-check verdict> triplet, where <post, image> are the text and image posted on a social media platform that contain some claim, and <fact-check verdict> is a fact-checking verdict determining whether the <post, image> spread misinformation and why. \nYour goal is to analyse the

In [ ]:
multimodal_interactor.

['edit', 'textual']

In [2]:
from pydantic import BaseModel, Field, create_model
from typing import Literal, List

# class Label(BaseModel):
#     """
#     Represents an a selected label from a taxonomy.
#     """

#     reasoning: str = Field(
#         ...,
#         description="The reason why the label was selected.",
#     )
#     label: Literal["yes", "no"] = Field(
#         ..., description="The name of the label."
#     )


# class Labels(BaseModel):
#     """
#     Represents a binary label for an image.
#     """

#     labels: List[Label] = Field(
#         ...,
#         description="A list of labels that were selected for the <post, image, fact-check verdict> triplet.",
#     )



def make_label_model(options: List[str]):
    MultimodalLabel = create_model(
        "MultimodalLabel",
        label=(
            Literal[tuple(options)],
            Field(..., description="The name of the label."),
        ),
        reasoning=(
            str,
            Field(..., description="The reason why the label was selected in a single sentence."),
        ),
    )
    MultimodalLabels = create_model(
        "MultimodalLabels",
        labels=(
            List[MultimodalLabel],
            Field(
                ...,
                description="A list of labels that were selected for the <post, image, fact-check verdict> triplet.",
            ),
        ),
    )
    return MultimodalLabels

schema = make_label_model(["yes", "no"])
schema.model_json_schema()


{'$defs': {'MultimodalLabel': {'properties': {'label': {'description': 'The name of the label.',
     'enum': ['yes', 'no'],
     'title': 'Label',
     'type': 'string'},
    'reasoning': {'description': 'The reason why the label was selected in a single sentence.',
     'title': 'Reasoning',
     'type': 'string'}},
   'required': ['label', 'reasoning'],
   'title': 'MultimodalLabel',
   'type': 'object'}},
 'properties': {'labels': {'description': 'A list of labels that were selected for the <post, image, fact-check verdict> triplet.',
   'items': {'$ref': '#/$defs/MultimodalLabel'},
   'title': 'Labels',
   'type': 'array'}},
 'required': ['labels'],
 'title': 'MultimodalLabels',
 'type': 'object'}

In [13]:
make_label_model(["yes", "no"]).model_json_schema()

{'$defs': {'Label': {'properties': {'label': {'description': 'The name of the label.',
     'enum': ['yes', 'no'],
     'title': 'Label',
     'type': 'string'},
    'reasoning': {'description': 'The reason why the label was selected in a single sentence.',
     'title': 'Reasoning',
     'type': 'string'}},
   'required': ['label', 'reasoning'],
   'title': 'Label',
   'type': 'object'}},
 'properties': {'labels': {'description': 'A list of labels that were selected for the <post, image, fact-check verdict> triplet.',
   'items': {'$ref': '#/$defs/Label'},
   'title': 'Labels',
   'type': 'array'}},
 'required': ['labels'],
 'title': 'Labels',
 'type': 'object'}

In [12]:
from atproto import FirehoseSubscribeReposClient, parse_subscribe_repos_message, models, CAR

client = FirehoseSubscribeReposClient()

def on_message_handler(message):
    commit = parse_subscribe_repos_message(message)
    if not isinstance(commit, models.ComAtprotoSyncSubscribeRepos.Commit):
        return
    
    if not commit.blocks:
        return
    
    car = CAR.from_bytes(commit.blocks)
    for op in commit.ops:
        if op.action in ["create"] and op.cid:
            data = car.blocks.get(op.cid)

            if data['$type'] == 'app.bsky.feed.post':
                text = data['text']

                if 'coffee' in text:
                    print(text)


client.start(on_message_handler)

KeyboardInterrupt: 

In [9]:
data

NameError: name 'data' is not defined